In [ ]:
import numpy as np
from tqdm import tqdm
from datetime import date, datetime
import xml.etree.ElementTree as ET
from joblib import Parallel, delayed
import tifffile as tf
import shutil
import glob
from pathlib import Path

import cv2
from skimage.restoration import rolling_ball
from skimage.exposure import rescale_intensity
import dask
import dask.array as da
from dask.diagnostics import ProgressBar
import time
import os 
import skimage

In [ ]:

overlapFrac = 1.52 # percentage of overlap between adjacent images. Measured empirically
frameY = 2016 # px
frameX = 2416 # px

In [ ]:
def extract_imaging_parameters(raw_tiles_path):
    xmlFile = [f for f in raw_tiles_path.rglob('*.xml') if 'ScanInfo' in f.stem]
    print(xmlFile)
    assert len(xmlFile) == 1

    tree = ET.parse(xmlFile[0])
    root = tree.getroot()

    channels = []
    for g1 in root.findall('Group'):
        if g1.attrib['Name'] == 'Experiment':
            for g2 in g1.findall('Settings'):
                for g3 in g2.findall('Setting'):

                    if g3.attrib['Parameter'] == 'ScanningEnabled':
                        assert g3.text == 'True'

                    if g3.attrib['Parameter'].startswith('Channel_'):
                        channels.append(g3.text)
                
        if g1.attrib['Name'] == 'Calibration':
            for g2 in g1.findall('Settings'):
                for g3 in g2.findall('Setting'):
                    if g3.attrib['Parameter'] == 'MicronsPerPixelX':
                        pixelSizeX = float(g3.text) # um per px

                    if g3.attrib['Parameter'] == 'MicronsPerPixelY':
                        pixelSizeY = float(g3.text)

        if g1.attrib['Name'] == 'Scan':
            for g2 in g1.findall('Settings'):
                for g3 in g2.findall('Setting'):

                    if g3.attrib['Parameter'] == 'Scan Width (mm)':
                        width = float(g3.text) * 1000 # mm to um
                    if g3.attrib['Parameter'] == 'Scan Height (mm)':
                        height = float(g3.text) * 1000 # mm to um

    tileX = int(np.ceil(width / (frameX * pixelSizeX)))
    tileY = int(np.ceil(height / (frameY * pixelSizeY)))
    
    return tileX, tileY, channels

In [ ]:
def rename_tiles(raw_tiles_path, tifFiles, fileNum, idx, ii):
    fileName = f'Tile{str(idx).zfill(6)}.tif'
    fileName = [f for f in tifFiles if f.name == fileName]
    assert len(fileName) == 1

    plane = tf.imread(fileName[0]) # YX

    # export and rename
    fileOut = Path(raw_tiles_path, f'Tile{str(fileNum).zfill(6)}_CH{ii + 1}.tif')
    tf.imwrite(fileOut, plane, photometric = 'minisblack')

    return None

In [ ]:
def format_tiles(channels, base_path, raw_tiles_path, tileX, tileY):

    if(len(channels) !=0):
        tifFiles = [f for f in raw_tiles_path.rglob('*.tif') if f.stem.startswith('Tile')]
        print(len(tifFiles))
        assert len(tifFiles) == tileX * tileY * len(channels)

        for ii, filter in enumerate(tqdm(channels)): # each channel

            tileRange = np.arange(0, len(tifFiles) // len(channels), 1) + 1
            tileIdx = np.arange(ii, len(tifFiles), len(channels)) + 1
                
            # compute tiles in parallel
            Parallel(n_jobs=-1, prefer = 'threads', verbose = 0)(delayed(rename_tiles)(raw_tiles_path, tifFiles, fileNum, idx, ii) for fileNum, idx in zip(tileRange, tileIdx))
    

In [ ]:
def write_macro(raw_tiles_path, stitched_img_path, channels, tileX, tileY, prev_macro_command=None):

    # write Fiji macro
    now = datetime.now() # current date and time
    date_time = now.strftime("%d%b%Y")
    macroName = Path(stitched_img_path, date_time + '_stitch.ijm')
    macro = open(macroName, 'w')


    if(len(channels) == 0):
        macro.write('run("MIST", "gridwidth=')
        macro.write(str(tileX))
        macro.write(' gridheight=')
        macro.write(str(tileY))
        macro.write(' starttile=1 imagedir=[')
        macro.write(str(raw_tiles_path).replace('\\', '/'))
        macro.write('] filenamepattern=Tile{pppppp}.jpg filenamepatterntype=SEQUENTIAL gridorigin=UL assemblefrommetadata=false assemblenooverlap=false globalpositionsfile=[] numberingpattern=VERTICALCOMBING startrow=0 startcol=0 extentwidth=')
        macro.write(str(tileX))
        macro.write(' extentheight=')
        macro.write(str(tileY))
        macro.write(' timeslices=0 istimeslicesenabled=false outputpath=[')
        macro.write(str(stitched_img_path).replace('\\', '/'))
        macro.write('] displaystitching=false outputfullimage=true outputmeta=true outputimgpyramid=false blendingmode=LINEAR blendingalpha=1.5 compressionmode=UNCOMPRESSED unit=MICROMETER unitx=1.0 unity=1.0 programtype=AUTO numcputhreads=32 loadfftwplan=true savefftwplan=true fftwplantype=MEASURE fftwlibraryname=libfftw3 fftwlibraryfilename=libfftw3.dll planpath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw\\fftPlans fftwlibrarypath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw stagerepeatability=0 horizontaloverlap=')
        macro.write(str(overlapFrac))
        macro.write(' verticaloverlap=')
        macro.write(str(overlapFrac))
        macro.write(' numfftpeaks=0 overlapuncertainty=NaN isusedoubleprecision=false isusebioformats=false issuppressmodelwarningdialog=true isenablecudaexceptions=false translationrefinementmethod=SINGLE_HILL_CLIMB numtranslationrefinementstartpoints=16 headless=false loglevel=MANDATORY debuglevel=NONE"); \n')


    else:

        for jj in range(len(channels)): # each channel

            if jj == 0: 
                # compute stitching
                macro.write('run("MIST", "gridwidth=')
                macro.write(str(tileX))
                macro.write(' gridheight=')
                macro.write(str(tileY))
                macro.write(' starttile=1 imagedir=[')
                macro.write(str(raw_tiles_path).replace('\\', '/'))
                macro.write('] filenamepattern=Tile{pppppp}_CH1.tif filenamepatterntype=SEQUENTIAL gridorigin=UL assemblefrommetadata=false assemblenooverlap=false globalpositionsfile=[] numberingpattern=VERTICALCOMBING startrow=0 startcol=0 extentwidth=')
                macro.write(str(tileX))
                macro.write(' extentheight=')
                macro.write(str(tileY))
                macro.write(' timeslices=0 istimeslicesenabled=false outputpath=[')
                macro.write(str(stitched_img_path).replace('\\', '/'))
                macro.write('] displaystitching=false outputfullimage=true outputmeta=true outputimgpyramid=false blendingmode=LINEAR blendingalpha=1.5 compressionmode=UNCOMPRESSED outfileprefix=CH')
                macro.write(str(jj + 1))
                macro.write('_ unit=MICROMETER unitx=1.0 unity=1.0 programtype=AUTO numcputhreads=32 loadfftwplan=true savefftwplan=true fftwplantype=MEASURE fftwlibraryname=libfftw3 fftwlibraryfilename=libfftw3.dll planpath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw\\fftPlans fftwlibrarypath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw stagerepeatability=0 horizontaloverlap=')
                macro.write(str(overlapFrac))
                macro.write(' verticaloverlap=')
                macro.write(str(overlapFrac))
                macro.write(' numfftpeaks=0 overlapuncertainty=NaN isusedoubleprecision=false isusebioformats=false issuppressmodelwarningdialog=true isenablecudaexceptions=false translationrefinementmethod=SINGLE_HILL_CLIMB numtranslationrefinementstartpoints=16 headless=false loglevel=MANDATORY debuglevel=NONE"); \n')

            else: # apply stitching to other channels
                macro.write('run("MIST", "gridwidth=')
                macro.write(str(tileX))
                macro.write(' gridheight=')
                macro.write(str(tileY))
                macro.write(' starttile=1 imagedir=[')
                macro.write(str(raw_tiles_path).replace('\\', '/'))
                macro.write('] filenamepattern=Tile{pppppp}_CH')
                macro.write(str(jj + 1))
                macro.write('.tif filenamepatterntype=SEQUENTIAL gridorigin=UL assemblefrommetadata=true assemblenooverlap=false globalpositionsfile=[')
                macro.write(str(stitched_img_path).replace('\\', '/') + '/CH1_global-positions-0.txt')
                macro.write('] numberingpattern=VERTICALCOMBING startrow=0 startcol=0 extentwidth=')
        
                #macro.write('.tif filenamepatterntype=SEQUENTIAL gridorigin=UL assemblefrommetadata=false assemblenooverlap=false globalpositionsfile=[] numberingpattern=VERTICALCOMBING startrow=0 startcol=0 extentwidth=')
                #macro.write(str(largeImagePath / 'img_t1_z1_c1_global-positions-0.txt').replace('\\', '/'))
                #macro.write('] numberingpattern=HORIZONTALCONTINUOUS startrow=0 startcol=0 extentwidth=')
                macro.write(str(tileX))
                macro.write(' extentheight=')
                macro.write(str(tileY))
                macro.write(' timeslices=0 istimeslicesenabled=false outputpath=[')
                macro.write(str(stitched_img_path).replace('\\', '/'))
                macro.write('] displaystitching=false outputfullimage=true outputmeta=true outputimgpyramid=false blendingmode=LINEAR blendingalpha=1.5 compressionmode=UNCOMPRESSED outfileprefix=CH')
                macro.write(str(jj + 1))
                macro.write('_ unit=MICROMETER unitx=1.0 unity=1.0 programtype=AUTO numcputhreads=32 loadfftwplan=true savefftwplan=true fftwplantype=MEASURE fftwlibraryname=libfftw3 fftwlibraryfilename=libfftw3.dll planpath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw\\fftPlans fftwlibrarypath=C:\\Users\\akumar702\\Fiji.app\\lib\\fftw stagerepeatability=0 horizontaloverlap=')
                macro.write(str(overlapFrac))
                macro.write(' verticaloverlap=')
                macro.write(str(overlapFrac))
                macro.write(' numfftpeaks=0 overlapuncertainty=NaN isusedoubleprecision=false isusebioformats=false issuppressmodelwarningdialog=true isenablecudaexceptions=false translationrefinementmethod=SINGLE_HILL_CLIMB numtranslationrefinementstartpoints=16 headless=false loglevel=MANDATORY debuglevel=NONE"); \n')

    if(prev_macro_command!=None):
        macro.write(prev_macro_command)

    macro.close()

    # print out command to run this macro in Fiji
    print('runMacro("' + str(Path(stitched_img_path, macro.name)).replace('\\', '/') + '");')
    return('runMacro("' + str(Path(stitched_img_path, macro.name)).replace('\\', '/') + '");')

In [ ]:
def stitch_tiles(macro_path, ij):
    ij.py.run_script('ijm', macro_path) 
    
    return None

In [ ]:
def rolling_ball_for_tile(tile, r):
    return tile-rolling_ball(tile, radius=r)

In [ ]:
def rescale_tile_intensity(tile, in_range):
    return rescale_intensity(tile, in_range=in_range, out_range=(0, 65000))

In [ ]:
def subtract_af(base_path, img_dict, split):

    start = time.time()
    print(base_path)
    idx = img_dict['base_path'].index(base_path)
    rbr = img_dict['rbr'][idx]
    empty_channel_index = img_dict['empty_channel_index'][idx]
    save_empty = img_dict['empty'][idx]

    if(img_dict['microscope'][idx] == 'Nikon'):
        stitched_folder_path =  base_path + '/02 TIF Stitched'
    elif(img_dict['microscope'][idx] == 'Echo_FOV'):
        stitched_folder_path = base_path
    elif(img_dict['microscope'][idx] == 'Echo_Multiple'):
        stitched_folder_path = base_path
    else:
        stitched_folder_path =  base_path + '/Stitched'
    bg_stitched_folder_path = base_path + '/Stitched_Background_Removed' 

    Path(bg_stitched_folder_path).mkdir(exist_ok=True)

    counter = 0

    tifFiles = []
    tifFiles_sizes = []

    while (len(tifFiles) != len(rbr) and len(list(set(tifFiles_sizes))) !=1):
        if(img_dict['microscope'][idx] == 'Echo_FOV' or img_dict['microscope'][idx] == 'Echo_Multiple'):
            tifFiles_out_of_order = [f for f in Path(stitched_folder_path).rglob('*.tif') if 'Merge' not in str(f)]
            print(tifFiles_out_of_order)
            tifFiles = []
            for channel in ['DAPI', 'FITC', 'TX RED', 'TX_RED', 'CY 5', 'CY_5']:
                match = next((f for f in tifFiles_out_of_order if channel in str(f)), None)
                
                if match is not None:
                    tifFiles.append(match)
        else:
            tifFiles = [f for f in Path(stitched_folder_path).rglob('*.tif')]
            tifFiles_sizes = [os.path.getsize(str(f)) for f in tifFiles] 
     
    if(len(tifFiles) > 0):
        print('tif files found') 
    
    print(tifFiles)


    channel_processed_list = []

    for file in tifFiles:
        if(rbr[counter]==-1):
            if(save_empty):
                empty_channel_path = tifFiles[empty_channel_index]
                empty_channel = tf.imread(empty_channel_path)
                if(len(empty_channel.shape) == 3):
                    empty_channel = skimage.color.rgb2gray(empty_channel)
                empty_channel_rescaled = da.from_array(empty_channel, chunks=(1000, 1000)).map_blocks(rescale_tile_intensity, (np.min(empty_channel), np.max(empty_channel)))
                del empty_channel
                channel_processed_list.append(empty_channel_rescaled)
                del empty_channel_rescaled
        
        elif(rbr[counter] != 0):
            
            channel = tf.imread(file)
            if(len(channel.shape) == 3):
                channel = skimage.color.rgb2gray(channel)
            channel_tile_list = np.array_split(channel, split)
            del channel 

            print('rolling ball for CH' + str(counter+1))
            channel_tile_bg_list = [] 
            for tile in channel_tile_list:
                tile_bg = dask.delayed(rolling_ball_for_tile)(tile, rbr[counter])
                channel_tile_bg_list.append(da.from_delayed(tile_bg, shape=tile.shape, dtype=tile.dtype))
            
            channel_bg_removed = da.concatenate(channel_tile_bg_list, axis=0)
            
            del channel_tile_bg_list

            channel_processed_list.append(channel_bg_removed)

            del channel_bg_removed
        
        else:
            channel = tf.imread(file)
            if(len(channel.shape) == 3):
                channel = skimage.color.rgb2gray(channel)
            channel_rescaled = da.from_array(channel, chunks = (1000, 1000)).map_blocks(rescale_tile_intensity, (np.min(channel), np.max(channel)))
            del channel 
            
            empty_channel_path = tifFiles[empty_channel_index]
            
            empty_channel = tf.imread(empty_channel_path)
            if(len(empty_channel.shape) == 3):
                empty_channel = skimage.color.rgb2gray(empty_channel)
            empty_channel_rescaled = da.from_array(empty_channel, chunks=(1000, 1000)).map_blocks(rescale_tile_intensity, (np.min(empty_channel), np.max(empty_channel)))
            del empty_channel

            if(empty_channel_rescaled.shape[0]>channel_rescaled.shape[0]):
                if(empty_channel_rescaled.shape[1]>channel_rescaled.shape[1]):
                    channel_rescaled = da.pad(channel_rescaled, ((0, empty_channel_rescaled.shape[0] - channel_rescaled.shape[0]), (0, empty_channel_rescaled.shape[1] - channel_rescaled.shape[1])))
                else:
                    channel_rescaled = da.pad(channel_rescaled, ((0, empty_channel_rescaled.shape[0] - channel_rescaled.shape[0]), (0, 0)))
                    empty_channel_rescaled = da.pad(empty_channel_rescaled, ((0, 0), (0, channel_rescaled.shape[1] - empty_channel_rescaled.shape[1])))
            else:
                if(channel_rescaled.shape[1]>empty_channel_rescaled.shape[1]):
                    empty_channel_rescaled = da.pad(empty_channel_rescaled, ((0, channel_rescaled.shape[0] - empty_channel_rescaled.shape[0]), (0, channel_rescaled.shape[1] - empty_channel_rescaled.shape[1])))
                else:
                    channel_rescaled = da.pad(channel_rescaled, ((0, 0), (0, empty_channel_rescaled.shape[1] - channel_rescaled.shape[1])))
                    empty_channel_rescaled = da.pad(empty_channel_rescaled, ((0, channel_rescaled.shape[0] - empty_channel_rescaled.shape[0]), (0, 0)))

            print('background subtraction for CH' + str(counter+1))

            channel_bg_removed = (
                channel_rescaled.astype(np.int32)
                - empty_channel_rescaled.astype(np.int32)
            )

            del channel_rescaled 
            del empty_channel_rescaled
                    
            print('clipping negative values for CH' + str(counter+1))
            channel_bg_removed = da.clip(channel_bg_removed, 0, 65535)
            channel_bg_removed = channel_bg_removed.astype(np.uint16)

            channel_processed_list.append(channel_bg_removed)
            
            del channel_bg_removed 
        
        print(file)

        counter+=1

    if(save_empty):
        channel_processed_idx = [i for i, x in enumerate(img_dict['rbr'][idx])]
    else:
        channel_processed_idx = [i for i, x in enumerate(img_dict['rbr'][idx]) if x >= 0]
    print(len(channel_processed_list))
    
    for i in range(len(channel_processed_list)):
        with ProgressBar():
            tf.imwrite(Path(base_path + '/Stitched_Background_Removed' + '/CH' + str(channel_processed_idx[i]+1)+ '_stitched_background_removed.ome.tif'), channel_processed_list[i].compute())


    del channel_processed_list
    end = time.time()
    print(end-start)

    return None


In [ ]:
img_dict = {}
img_dict['base_path'] = [r"", 
                         r""] #path to processed images 

img_dict['rbr'] = [[50, -1, 50], 
                   [50, -1, 50]] #CHANGE

img_dict['empty_channel_index'] = [1, 1] #CHANGE 

img_dict['microscope'] = ['Echo_Multiple']*len(img_dict['base_path'])

img_dict['empty'] = [True, True]


In [ ]:
macro_commands = []
i = 0
for base_path_str in img_dict["base_path"]:
    
    base_path = Path(base_path_str)
    print(base_path)
    assert base_path.exists()
    
    raw_tiles_folder =  'Raw'
    raw_tiles_path = base_path / raw_tiles_folder

    # folder to saved stitched images
    stitched_img_path = base_path / 'Stitched'
    stitched_img_path.mkdir(exist_ok=True)

    tileX, tileY, channels = extract_imaging_parameters(raw_tiles_path)
    print(tileX)
    print(tileY)
    print(channels)
    
    format_tiles(channels, base_path, raw_tiles_path, tileX, tileY)
    
    if(img_dict["base_path"].index(base_path_str) !=0):
        print(img_dict["base_path"].index(base_path_str)-1)
        macro_command = write_macro(raw_tiles_path, stitched_img_path, channels, tileX, tileY, macro_commands[img_dict["base_path"].index(base_path_str)-1])
        macro_commands.append(macro_command)
    else:
        macro_command = write_macro(raw_tiles_path, stitched_img_path, channels, tileX, tileY)
        macro_commands.append(macro_command)
   
    i+=1 


In [ ]:
for base_path_str in img_dict["base_path"]:
   subtract_af(base_path_str, img_dict, 100)